From the `Bamboo_setup` directory, run the following command:
```bash
jupyter notebook --no-browser --port=8888
```
Copy the URL starting with `http://localhost:8888/`.
When picking the kernel for this notebook, click **Existing Jupyter Server** and paste the URL.
Name your server 'localhost'.
From localhost, select the 'Python 3' kernel.

In [1]:
import sys
from pathlib import Path
BAMBOO_SETUP = Path.cwd()
NN_POSTPROCESSING = BAMBOO_SETUP / 'src' / 'post_processing' / 'NN'
NOTEBOOKS = NN_POSTPROCESSING / 'notebooks'
sys.path.append(str((BAMBOO_SETUP/'src').resolve()))
import pandas as pd
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
# pd.set_option('display.max_colwidth', None)  # Allow columns to be fully displayed
%load_ext autoreload

In [6]:
%autoreload 2

# Setup a data handler

In [2]:
from post_processing.NN.DataHandler import DataHandler
datahandler = DataHandler(
    workdir=Path('/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0822/LLR_and_vars_4o5'),
    tree_name='SL_res_2b_x',
    total_inputs= NN_POSTPROCESSING / 'input/8llrs1D.txt'
)
df = datahandler.load_data()
df = datahandler.fix_any_mismatch(df)
df

Welcome to JupyROOT 6.30/02
	Loading data...


,event,genWeight,bjet0_pt_llr,bjets_dEta_llr,bjets_dPhi_llr,bjets_dR_llr,bjets_mbb_llr,mjj_llr,trijet_mInv_llr,trijet_pt_rat_llr,File,Process
186577,18,3.805950,-0.278,0.401,0.742,1.496,1.167,0.428,0.439,0.014,tbarWplus_dl,tW
13167,32,0.033119,-0.242,0.329,0.937,1.507,0.849,0.393,0.549,1.259,bbWW_sl,HH_bbWW
706510,36,81.103897,-0.322,-0.791,-1.330,-2.183,-2.512,-0.028,0.317,-0.649,TTbar_dl,ttbar
13168,42,0.033119,0.132,0.239,1.084,1.504,0.714,0.130,0.117,-0.098,bbWW_sl,HH_bbWW
11905,58,0.033119,0.109,0.358,0.747,1.493,1.051,-0.491,0.396,-0.674,bbWW_dl,HH_bbWW
...,...,...,...,...,...,...,...,...,...,...,...,...
30323,744899478,83745.546875,-0.288,0.327,-0.377,-0.090,-0.269,-0.151,-0.445,-0.622,Wjets_2J,WJets
30340,745108688,83745.546875,-0.223,0.343,0.757,1.520,0.263,0.193,0.558,-0.775,Wjets_2J,WJets
30318,745244862,83745.546875,-0.219,NaN,0.886,-1.550,-inf,-0.071,-1.004,0.563,Wjets_2J,WJets
30006,745282826,-83745.546875,-0.325,-0.230,0.668,-0.090,0.155,-0.419,-0.551,-0.246,Wjets_2J,WJets


In [14]:
df = df[['event', 'genWeight', 'bjet0_pt_llr', 'bjets_dEta_llr', 'bjets_dPhi_llr', 'bjets_dR_llr', 'bjets_mbb_llr', 'mjj_llr', 'trijet_mInv_llr', 'trijet_pt_rat_llr']]
df

,event,genWeight,bjet0_pt_llr,bjets_dEta_llr,bjets_dPhi_llr,bjets_dR_llr,bjets_mbb_llr,mjj_llr,trijet_mInv_llr,trijet_pt_rat_llr
186577,18,3.805950,-0.278,0.401,0.742,1.496,1.167,0.428,0.439,0.014
13167,32,0.033119,-0.242,0.329,0.937,1.507,0.849,0.393,0.549,1.259
706510,36,81.103897,-0.322,-0.791,-1.330,-2.183,-2.512,-0.028,0.317,-0.649
13168,42,0.033119,0.132,0.239,1.084,1.504,0.714,0.130,0.117,-0.098
11905,58,0.033119,0.109,0.358,0.747,1.493,1.051,-0.491,0.396,-0.674
...,...,...,...,...,...,...,...,...,...,...
30323,744899478,83745.546875,-0.288,0.327,-0.377,-0.090,-0.269,-0.151,-0.445,-0.622
30340,745108688,83745.546875,-0.223,0.343,0.757,1.520,0.263,0.193,0.558,-0.775
30318,745244862,83745.546875,-0.219,NaN,0.886,-1.550,-inf,-0.071,-1.004,0.563
30006,745282826,-83745.546875,-0.325,-0.230,0.668,-0.090,0.155,-0.419,-0.551,-0.246


# Inspect preprocessing

In [16]:
def check(df):
    nan_cells = df.isna().sum().sum()
    neg_9999_cells = (df == -9999).sum().sum()
    neg_9_cells = (df == -9).sum().sum()
    pos_9999_cells = (df == 9999).sum().sum()

    print(f"Number of cells with NaN: {nan_cells}")
    print(f"Number of cells with -9999: {neg_9999_cells}")
    print(f"Number of cells with -9: {neg_9_cells}")
    print(f"Number of cells with +9999: {pos_9999_cells}")

In [17]:
df.isna().sum().to_frame().T

,event,genWeight,bjet0_pt_llr,bjets_dEta_llr,bjets_dPhi_llr,bjets_dR_llr,bjets_mbb_llr,mjj_llr,trijet_mInv_llr,trijet_pt_rat_llr
0,0,0,3492,32357,0,571,2749,0,0,4969


In [16]:
total_rows_containing_nans = df.isna().any(axis=1).sum()
print(f"Events containing nans: {total_rows_containing_nans}")
df_nan = df[df.isna().any(axis=1)]
df_nan

Events containing nans: 43828


,event,genWeight,bjet0_pt_llr,bjets_dEta_llr,bjets_dPhi_llr,bjets_dR_llr,bjets_mbb_llr,mjj_llr,trijet_mInv_llr,trijet_pt_rat_llr
706511,106,81.103897,-0.243,NaN,-1.184,-3.566,-0.517,-0.283,-0.414,0.596
706517,338,81.103897,0.179,NaN,-1.187,-3.089,-1.365,0.149,0.230,-0.046
258821,1692,36.054901,0.232,NaN,-0.690,-2.273,-inf,0.337,-0.689,-0.534
12779,2308,0.033119,NaN,0.239,-1.322,-1.053,-inf,-0.186,0.657,-0.585
567232,3254,81.103897,-0.313,NaN,0.886,-1.505,0.489,0.382,0.111,0.008
...,...,...,...,...,...,...,...,...,...,...
28847,740587658,83745.546875,-0.325,-0.950,0.549,-0.645,0.959,0.454,-0.099,NaN
28584,740747006,-83745.546875,NaN,-0.190,-1.354,-1.671,-inf,0.286,-1.287,-0.780
28401,741401132,83745.546875,-0.263,NaN,-0.916,-3.407,-0.607,0.063,-0.255,-0.600
30005,744511582,83745.546875,-0.296,NaN,0.608,-1.370,-1.527,0.197,0.345,0.665
